# BoneTest: EfficientNet-B3 Sex-Fusion Training Pipeline (RSNA Pediatric Bone Age)

This notebook trains an ensemble member of the **EfficientNetB3SexFusion** model on GPU (Kaggle T4/P100 or Colab).

### Pipeline Highlights:
1. **Pre-processing**: Otsu Hand ROI crop + CLAHE + $384 \times 384$ aspect-preserving letterbox
2. **Backbone**: ImageNet-pretrained EfficientNet-B3 with 3-channel replicated grayscale
3. **Demographic Fusion**: 32-dimensional learnable sex embedding concatenated before the projection head
4. **Optimization**: Huber Loss ($\delta = 1.0$) on standardized age targets, AdamW, Cosine Annealing with warmup
5. **Auditing**: Git hash, manifest SHA-256, and normalization metadata embedded into every checkpoint

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
import os
import sys
import time
import math
import json
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torch.optim import AdamW
from torch.optim.lr_scheduler import LambdaLR

print(f"PyTorch: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")

## 1. Model Definition: EfficientNetB3SexFusion

In [ ]:
class EfficientNetB3SexFusion(nn.Module):
    def __init__(self, pretrained: bool = True, sex_embed_dim: int = 32, dropout: float = 0.2) -> None:
        super().__init__()
        weights = models.EfficientNet_B3_Weights.DEFAULT if pretrained else None
        base = models.efficientnet_b3(weights=weights)
        
        self.features = base.features
        self.avgpool = base.avgpool
        in_features = base.classifier[1].in_features  # 1536 for B3
        
        self.sex_embedding = nn.Embedding(num_embeddings=2, embedding_dim=sex_embed_dim)
        fused_dim = in_features + sex_embed_dim
        
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(fused_dim, 256),
            nn.SiLU(),
            nn.Dropout(p=dropout / 2),
            nn.Linear(256, 1),
        )
        
    def forward(self, images: torch.Tensor, sex: torch.Tensor) -> torch.Tensor:
        img_feats = self.features(images)
        pooled = self.avgpool(img_feats)
        flat_img = torch.flatten(pooled, 1)
        sex_feats = self.sex_embedding(sex)
        fused = torch.cat([flat_img, sex_feats], dim=1)
        out = self.head(fused)
        return out.squeeze(-1)

## 2. Dataset & Preprocessing

In [ ]:
def crop_hand_roi(image: np.ndarray, margin_pct: float = 0.05, min_area_ratio: float = 0.15) -> np.ndarray:
    blurred = cv2.GaussianBlur(image, (5, 5), 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(thresh, connectivity=8)
    
    if num_labels <= 1:
        return image
    
    areas = stats[1:, cv2.CC_STAT_AREA]
    largest_idx = int(np.argmax(areas)) + 1
    largest_area = stats[largest_idx, cv2.CC_STAT_AREA]
    h_img, w_img = image.shape
    
    if largest_area < (h_img * w_img * min_area_ratio):
        return image
        
    x = stats[largest_idx, cv2.CC_STAT_LEFT]
    y = stats[largest_idx, cv2.CC_STAT_TOP]
    w = stats[largest_idx, cv2.CC_STAT_WIDTH]
    h = stats[largest_idx, cv2.CC_STAT_HEIGHT]
    
    pad_x = int(w * margin_pct)
    pad_y = int(h * margin_pct)
    
    x0 = max(0, x - pad_x)
    y0 = max(0, y - pad_y)
    x1 = min(w_img, x + w + pad_x)
    y1 = min(h_img, y + h + pad_y)
    
    return image[y0:y1, x0:x1]

def apply_clahe(image: np.ndarray, clip_limit: float = 2.0, tile_grid_size: tuple = (8, 8)) -> np.ndarray:
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    return clahe.apply(image)

def resize_letterbox(image: np.ndarray, target_size: int = 384) -> np.ndarray:
    h, w = image.shape[:2]
    scale = target_size / max(h, w)
    new_w, new_h = int(w * scale), int(h * scale)
    resized = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_AREA)
    
    canvas = np.zeros((target_size, target_size), dtype=np.uint8)
    y_off = (target_size - new_h) // 2
    x_off = (target_size - new_w) // 2
    canvas[y_off:y_off + new_h, x_off:x_off + new_w] = resized
    return canvas

def preprocess_radiograph(raw_img: np.ndarray, target_size: int = 384) -> np.ndarray:
    roi = crop_hand_roi(raw_img)
    enhanced = apply_clahe(roi)
    return resize_letterbox(enhanced, target_size=target_size)

## 3. Training Execution (e.g. Seed 42, 1337, or 2026)

In [ ]:
# Set seed and hyperparameters
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BATCH_SIZE = 32
EPOCHS = 30
LR = 3e-4
WEIGHT_DECAY = 1e-4

print(f"Training configuration initialized with Seed {SEED}.")